In [1]:
from pathlib import Path

PARENT = Path.cwd().parent
print(PARENT)


c:\Users\hyunj\Seoul_Strolling_Adventure


In [2]:
import os, re
import numpy as np
import pandas as pd

# ===== 설정 =====
IN_USAGE_XLSX = fr"{PARENT}\tourism_transport_mapper\대중교통이용객\읍면동 최종 이용량_통합.xlsx"
IN_POI_CSV    = fr"{PARENT}\tourism_transport_mapper\지역\df_sorted.csv"
IN_STORE_CSV  = fr"{PARENT}\Crawling\전국_그룹분리_csv\_ckpt\store_scores.csv"
IN_REGION_CSV = fr"{PARENT}\tourism_transport_mapper\대중교통위치\tourist_spots_with_addresses.csv" #전국.xlsx_주소 업데이트한 csv
IN_TRANSPORT_CSV = fr"{PARENT}\tourism_transport_mapper\대중교통위치\관광지_대중교통_매칭최종.xlsx"

trans = pd.read_excel(IN_USAGE_XLSX)
tour_mapping = pd.read_csv(IN_POI_CSV)
store_scores = pd.read_csv(IN_STORE_CSV)
region_adress = pd.read_csv(IN_REGION_CSV)
transport = pd.read_excel(IN_TRANSPORT_CSV)

store_scores.head(2)
# tour_mapping.head(2)
# region.head(2)
# transport.head(2)
# trans.head(2)

,region,store_name,address,count,avg_sentiment,avg_class_prob,count_norm,review_score
0,대구,(주)신세계 동대구복합환승센터,대구광역시,17,0.632353,0.299871,0.34,0.424075
1,대구,(주)즐거운세상인터불고호텔,대구광역시 (주)즐거운세상,2,0.625000,0.327661,0.04,0.330887


In [3]:
transport.head(2)

,title,addr1,cat1,cat2,cat3,mapx,mapy,closest_subway_station,closest_subway_line,closest_bus_station,정류장명,정류장번호
0,'연천 1급수' 계곡 따라 강 따라,경기 연천군 연천읍 고문리 산 21,추천코스,힐링코스,힐링코스,127.128632,38.070370,연천역,['경원선'],재인폭포,재인폭포,['GGB238000228']
1,'태양의 후예' 촬영지 여행 코스,경기 파주시 군내면 백연리 산 80,추천코스,가족코스,가족코스,126.729784,37.896789,임진강역,['경의중앙선'],적십자사,적십자사,"['GGB229000057', 'GGB229000072']"


In [3]:
import re
import unicodedata as ud

# --- 1) 시도명 정규화 매핑 (네가 준 dict + 보강) ---
sido_map = {
    '서울': '서울특별시', '서울특별시': '서울특별시', '서울시': '서울특별시',
    '부산': '부산광역시', '부산광역시': '부산광역시',
    '대구': '대구광역시', '대구광역시': '대구광역시',
    '인천': '인천광역시', '인천광역시': '인천광역시',
    '광주': '광주광역시', '광주광역시': '광주광역시',
    '대전': '대전광역시', '대전광역시': '대전광역시',
    '울산': '울산광역시', '울산광역시': '울산광역시', '울산시': '울산광역시',
    '세종': '세종특별자치시', '세종특별자치시': '세종특별자치시',
    '경기': '경기도', '경기도': '경기도',
    '강원': '강원도', '강원도': '강원도', '강원특별자치도': '강원도',
    '충남': '충청남도', '충청남도': '충청남도',
    '충북': '충청북도', '충청북도': '충청북도',
    '전남': '전라남도', '전라남도': '전라남도',
    '전북': '전라북도', '전라북도': '전라북도', '전북특별자치도': '전라북도',
    '경남': '경상남도', '경상남도': '경상남도',
    '경북': '경상북도', '경상북도': '경상북도',
    '제주': '제주특별자치도', '제주도': '제주특별자치도', '제주특별자치도': '제주특별자치도',
}

def norm_sido(x: str) -> str:
    if pd.isna(x): return x
    x = ud.normalize("NFC", str(x)).strip()
    return sido_map.get(x, x)

def extract_sido_from_addr(addr: str) -> str:
    if pd.isna(addr): return addr
    addr = ud.normalize("NFC", str(addr)).strip()
    # 주소 맨 앞 토큰(시/도 명칭) 추출
    m = re.match(r'^([^\s]+)', addr)
    raw = m.group(1) if m else addr.split()[0]
    return norm_sido(raw)

# --- 2) store_scores: region 약칭 -> 풀네임, (region_full, store_name) 단일화 ---
sc = store_scores.copy()
sc['region_full'] = sc['region'].apply(norm_sido)

# 주소가 여러 개인 동일 상호가 있을 수 있으니 (region_full, store_name)로 review_score 최대값 집계
if 'review_score' not in sc.columns:
    raise ValueError("store_scores에 'review_score' 컬럼이 필요합니다.")
sc_key = (sc.groupby(['region_full','store_name'], as_index=False)['review_score']
            .max()  # 필요 시 mean()으로 바꿔도 됨
         )

# --- 3) tour_mapping: addr1에서 시도 추출/정규화 ---
tm = tour_mapping.copy()
tm['sido_norm'] = tm['addr1'].apply(extract_sido_from_addr)

# --- 4) (sido_norm, title) ↔ (region_full, store_name) 매칭으로 review_score 붙이기 ---
merged = tm.merge(
    sc_key,
    left_on=['sido_norm','title'],
    right_on=['region_full','store_name'],
    how='left',  # 매칭 안 되면 NaN
    validate='m:1'  # tour_mapping 한 title/sido에 대해 최대 1개 review_score로 기대
)

# --- 5) 결과 컬럼만 슬림화 (title, addr1, tour_score, review_score) ---
need_cols = ['title','addr1','tour_score','review_score']
# 혹시 'tour_score' 오타/부재 방지
if 'tour_score' not in merged.columns:
    # 흔한 오타 방어 (없으면 KeyError)
    alt = [c for c in merged.columns if c.startswith('tour')][0]
    merged = merged.rename(columns={alt: 'tour_score'})

tour_mapping_final = merged[need_cols].copy()

# (선택) review_score 없던 행은 0으로
# tour_mapping_final['review_score'] = tour_mapping_final['review_score'].fillna(0.0)

tour_mapping_final.head()

,title,addr1,tour_score,review_score
0,모담 현대백화점킨텍스점,경기도 고양시 일산서구 호수로 817 (대화동),8.0238,NaN
1,장독대김치찜김치찌개_용산점,서울특별시 용산구 한강대로23길 55 (한강로3가),7.8077,NaN
2,경성한우불고기 송도컨벤시아 본점,인천광역시 연수구 센트럴로 123 송도컨벤시아,7.7990,0.640433
3,식물학 아이파크몰점,서울특별시 용산구 한강대로23길 55 (한강로3가),7.7830,NaN
4,고든램지버거,서울특별시 송파구 올림픽로 300 롯데월드타워앤드롯데월드몰,7.7829,NaN


In [4]:
# 병합
df = pd.merge(tour_mapping_final, transport, on=['title', 'addr1'], how='inner')

In [ ]:
# import pandas as pd
# from pathlib import Path

# # 병합
# df = pd.merge(tour_mapping_final, transport, on=['title', 'addr1'], how='inner')

# # review_score가 NaN인 경우 0으로 대체
# df['review_score'] = df['review_score'].fillna(0)
# df = df.drop(columns=["정류장명"])


# # 1) 그룹 키 결정 (contentid > title+addr1 > title > 첫 컬럼)
# if {"title","addr1"}.issubset(df.columns):
#     GROUP_KEYS = ["title","addr1"]
# elif "title" in df.columns:
#     GROUP_KEYS = ["title"]
# else:
#     GROUP_KEYS = [df.columns[0]]

# # 2) 모든 컬럼 사용 + '정류장번호'만 리스트 집계
# def _valid(x):
#     if pd.isna(x): 
#         return False
#     xs = str(x).strip()
#     return bool(xs) and xs.lower() != "nan"

# def first_non_null(s: pd.Series):
#     # 문자열/숫자 혼재해도 첫 유효값 반환
#     for v in s:
#         if _valid(v):
#             return v
#     return None

# def unique_list_keep_order(s: pd.Series):
#     seen, out = set(), []
#     for v in s:
#         if not _valid(v):
#             continue
#         vv = str(v).strip()
#         if vv not in seen:
#             seen.add(vv); out.append(vv)
#     return out

# def to_pylist_string(lst):
#     return "[" + ", ".join(f"'{x}'" for x in lst) + "]"

# agg = {}
# for c in df.columns:
#     if c in GROUP_KEYS:
#         continue
#     if c == "정류장번호":
#         agg[c] = unique_list_keep_order     # <-- 오직 이 컬럼만 리스트
#     else:
#         agg[c] = "first"                    # <-- 나머지는 대표값 하나(중복 제거 효과)

# df = df.groupby(GROUP_KEYS, as_index=False).agg(agg)

# # 리스트를 문자열로 변환
# if "정류장번호" in df.columns:
#     df["정류장번호"] = df["정류장번호"].apply(unique_list_keep_order).apply(to_pylist_string)

# # 확인
# df.head(3)

,title,addr1,tour_score,review_score,cat1,cat2,cat3,mapx,mapy,closest_subway_station,closest_subway_line,closest_bus_station,정류장번호
0,'연천 1급수' 계곡 따라 강 따라,경기 연천군 연천읍 고문리 산 21,3.5400,0.0,추천코스,힐링코스,힐링코스,127.128632,38.070370,연천역,['경원선'],재인폭포,['['GGB238000228']']
1,'태양의 후예' 촬영지 여행 코스,경기 파주시 군내면 백연리 산 80,3.7127,0.0,추천코스,가족코스,가족코스,126.729784,37.896789,임진강역,['경의중앙선'],적십자사,"['['GGB229000057', 'GGB229000072']']"
2,"(강릉~고성) 풍경 따라 맛 따라, 7번 국도 드라이브",강원특별자치도 강릉시 연곡면 영진리 72-21,3.5422,0.0,추천코스,힐링코스,힐링코스,128.845159,37.868829,춘천역,['경춘선'],주문진시외버스터미널,"['['TSB252000407', 'TSB252000979']']"


In [5]:
# -*- coding: utf-8 -*-
from config import KAKAO_API_KEY
import os
import requests
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# --- 설정 ---
OUTPUT_CSV_PATH = "관광지_법정동_매핑결과.csv"  # <--- 최종 결과 및 중간 저장 파일 경로
SAVE_INTERVAL = 500                           # <--- 500개 처리할 때마다 중간 저장

WORKERS       = 32      # 병렬 요청 스레드 수
REQ_DELAY     = 0.03    # 요청 간 최소 delay (rate limit 고려)
TIMEOUT_SEC   = 5

# ==============================
# 카카오 API 호출 함수
# ==============================
def kakao_coord2region(lon: float, lat: float) -> str:
    """
    경도(lon), 위도(lat) → 법정동명
    """
    # ... (함수 내용은 이전과 동일) ...
    url = "https://dapi.kakao.com/v2/local/geo/coord2regioncode.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    params = {"x": lon, "y": lat}
    try:
        r = requests.get(url, headers=headers, params=params, timeout=TIMEOUT_SEC)
        r.raise_for_status()
        data = r.json()
        docs = data.get("documents", [])
        if docs:
            for d in docs:
                if d.get("region_type") == "B":
                    return d.get("address_name")
        return None
    except:
        return None

# ==============================
# 메인 처리
# ==============================

# <--- 1. 기존 결과 파일이 있는지 확인하고 데이터 불러오기 ---
if os.path.exists(OUTPUT_CSV_PATH):
    print(f"'{OUTPUT_CSV_PATH}' 파일이 존재합니다. 중단된 지점부터 이어서 작업을 시작합니다.")
    df = pd.read_csv(OUTPUT_CSV_PATH, encoding="utf-8-sig")
else:
    print(f"새로운 작업을 시작합니다.")
    df["법정동"] = None # 법정동 컬럼 추가

# <--- 2. 아직 처리되지 않은 행만 대상으로 선택 ---
indices_to_process = df[df['법정동'].isna()].index
print(f"총 {len(df)}개 중 {len(indices_to_process)}개의 행에 대해 법정동 조회를 시작합니다.")


def _task(idx, lon, lat):
    if pd.isna(lon) or pd.isna(lat):
        return idx, None
    addr = kakao_coord2region(lon, lat)
    time.sleep(REQ_DELAY)
    return idx, addr

if indices_to_process.empty:
    print("모든 행의 법정동 조회가 이미 완료되었습니다.")
else:
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        # <--- 3. 처리할 인덱스만 작업으로 제출 ---
        futures = [ex.submit(_task, i, df.at[i, "mapx"], df.at[i, "mapy"]) for i in indices_to_process]
        
        completed_count = 0
        for fut in tqdm(as_completed(futures), total=len(indices_to_process), desc="법정동 조회 중"):
            idx, addr = fut.result()
            df.loc[idx, '법정동'] = addr
            completed_count += 1
            
            # <--- 4. 주기적으로 중간 결과 저장 ---
            if completed_count % SAVE_INTERVAL == 0:
                df.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")
                # tqdm.write(f"[{completed_count} / {len(indices_to_process)}] 중간 결과를 저장했습니다.") # 진행바와 겹치지 않게 로그 출력

# <--- 5. 모든 작업 완료 후 최종 저장 ---
print("\n모든 작업이 완료되었습니다. 최종 결과를 저장합니다.")
df.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")
df.head(3)

새로운 작업을 시작합니다.
총 50159개 중 50159개의 행에 대해 법정동 조회를 시작합니다.


법정동 조회 중: 100%|██████████| 50159/50159 [02:31<00:00, 332.13it/s]



모든 작업이 완료되었습니다. 최종 결과를 저장합니다.


,title,addr1,tour_score,review_score,addr2,areacode,booktour,cat1,cat2,cat3,...,mlevel,modifiedtime,sigungucode,tel,zipcode,closest_subway_station,closest_subway_line,closest_bus_station,closest_bus_id,법정동
0,모담 현대백화점킨텍스점,경기도 고양시 일산서구 호수로 817 (대화동),8.0238,NaN,8층,31.0,NaN,음식,음식점,한식,...,6.0,20241119170134,2.0,NaN,10391,주엽역,['일산선'],현대백화점,GGB219000625,경기도 고양시 일산서구 대화동
1,장독대김치찜김치찌개_용산점,서울특별시 용산구 한강대로23길 55 (한강로3가),7.8077,NaN,테이스티파크 4층,1.0,NaN,음식,음식점,한식,...,6.0,20250313144702,21.0,02-2012-0422,04377,신용산역,['4호선'],용산역,GGB102000161,서울특별시 용산구 한강로3가
2,경성한우불고기 송도컨벤시아 본점,인천광역시 연수구 센트럴로 123 송도컨벤시아,7.7990,0.640433,남문 1층,2.0,NaN,음식,음식점,한식,...,6.0,20250120101836,8.0,NaN,21998,인천대입구역,['인천지하철 1호선'],송도더샵퍼스트월드,ICB164000381,인천광역시 연수구 송도동


In [6]:
rows, cols = df.shape
print(f"행: {rows}, 열: {cols}")

행: 50159, 열: 28


In [7]:
# -*- coding: utf-8 -*-
# Kakao Local API로 관광지(좌표/주소/키워드) → 법정동 매핑
# ✅ 법정동이 빈칸(공백 포함)인 행만 처리
# - 단계별 마스크(법정동 isna) 적용
# - 주기 저장
# - QPS 레이트리밋 / 재시도
# - 좌표 지터링(거친→정밀)

import os, math, time, threading, requests, re
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
from typing import Optional, Tuple
from tqdm import tqdm
from config import KAKAO_API_KEY

# =========================
# 경로 / 저장 주기
# =========================
TARGET_CSV_PATH = "관광지_법정동_매핑결과.csv"
SAVE_INTERVAL   = 10

# =========================
# 튜닝 포인트
# =========================
WORKERS   = 12
QPS       = 15
TIMEOUT   = 6
MAX_RETRY = 4

# 2단계 지터링 파라미터
COARSE_STEP_M = 10      # 1단계 간격
COARSE_MAX_M  = 100     # 1단계 최대 반경
FINE_STEP_M   = 1       # 2단계 간격
FINE_MAX_M    = 100     # 2단계 최대 반경

OFFSETS_4 = [(1,0),(-1,0),(0,1),(0,-1)]
OFFSETS_8 = [(1,0),(-1,0),(0,1),(0,-1),(1,1),(1,-1),(-1,1),(-1,-1)]

# =========================
# 유틸
# =========================
def _is_wgs84(lon, lat):
    try:
        lon = float(lon); lat = float(lat)
        return (124 <= lon <= 132) and (33 <= lat <= 39)
    except Exception:
        return False

def _clean_address(s: Optional[str]) -> Optional[str]:
    if not isinstance(s, str): return None
    s = s.strip()
    if not s: return None
    s = re.sub(r'\([^)]*\)', '', s)                                   # 괄호 제거
    s = re.sub(r'\d+층|\d+호|\d+F|\btel[:\s]*\d.*$', '', s, flags=re.I) # 층/호/TEL 제거
    return s.strip() or None

def _m_to_deg_lat(m): return m / 111_320.0
def _m_to_deg_lon(m, lat_deg): return m / (111_320.0 * math.cos(math.radians(lat_deg)))

# =========================
# 세션 + 레이트리밋 + 재시도
# =========================
SESSION = requests.Session()
SESSION.headers.update({"Authorization": f"KakaoAK {KAKAO_API_KEY}"})

class RateLimiter:
    def __init__(self, qps: int):
        self.min_interval = 1.0 / max(1, qps)
        self.lock = threading.Lock()
        self.last = 0.0
    def wait(self):
        with self.lock:
            now = time.time()
            wait = self.min_interval - (now - self.last)
            if wait > 0:
                time.sleep(wait)
            self.last = time.time()

limiter = RateLimiter(QPS)

def _call_json(url: str, params: dict) -> Tuple[Optional[dict], Optional[str]]:
    """공통 호출기: (json, error_reason)"""
    for attempt in range(MAX_RETRY):
        try:
            limiter.wait()
            r = SESSION.get(url, params=params, timeout=TIMEOUT)
            if r.status_code == 429:
                time.sleep(min(2 ** attempt, 8));  continue
            if r.status_code >= 500:
                time.sleep(min(1.5 ** attempt, 6));  continue
            r.raise_for_status()
            return r.json(), None
        except requests.RequestException:
            time.sleep(min(1.5 ** attempt, 6))
    return None, "HTTP_FAIL"

# =========================
# Kakao API 래퍼
# =========================
_coord_cache: dict[tuple[float,float], tuple[Optional[str], Optional[str]]] = {}
_addr_cache: dict[str, tuple[Optional[tuple[float,float]], Optional[str]]] = {}

def kakao_coord2region(lon: float, lat: float) -> Tuple[Optional[str], Optional[str]]:
    """좌표 -> (법정동명 or None, fail_reason or None)"""
    key = (round(float(lon),7), round(float(lat),7))
    if key in _coord_cache: return _coord_cache[key]
    url = "https://dapi.kakao.com/v2/local/geo/coord2regioncode.json"
    js, err = _call_json(url, {"x": key[0], "y": key[1]})
    if js is None:
        _coord_cache[key] = (None, err);  return _coord_cache[key]
    docs = js.get("documents", [])
    b = next((d for d in docs if d.get("region_type")=="B"), None)
    if b:
        _coord_cache[key] = (b.get("address_name"), None);  return _coord_cache[key]
    h = next((d for d in docs if d.get("region_type")=="H"), None)
    if h:
        _coord_cache[key] = (h.get("address_name"), "H_ONLY");  return _coord_cache[key]
    _coord_cache[key] = (None, "EMPTY");  return _coord_cache[key]

def kakao_address_to_coord(address: str) -> Tuple[Optional[tuple[float,float]], Optional[str]]:
    """주소 -> (lon,lat)"""
    address = _clean_address(address)
    if not address: return None, "ADDR_EMPTY"
    if address in _addr_cache: return _addr_cache[address]
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    js, err = _call_json(url, {"query": address})
    if js is None:
        _addr_cache[address] = (None, err);  return _addr_cache[address]
    docs = js.get("documents", [])
    if not docs:
        _addr_cache[address] = (None, "NO_MATCH");  return _addr_cache[address]
    x, y = docs[0].get("x"), docs[0].get("y")
    if not x or not y:
        _addr_cache[address] = (None, "NO_XY");  return _addr_cache[address]
    _addr_cache[address] = ((float(x), float(y)), None);  return _addr_cache[address]

def kakao_keyword_to_coord(keyword: str, bias: Optional[tuple[float,float]]=None) -> Tuple[Optional[tuple[float,float]], Optional[str]]:
    """키워드(지명/시설명) -> 좌표"""
    keyword = _clean_address(keyword)
    if not keyword: return None, "KW_EMPTY"
    url = "https://dapi.kakao.com/v2/local/search/keyword.json"
    params = {"query": keyword}
    if bias:
        params.update({"x": bias[0], "y": bias[1], "radius": 5000})
    js, err = _call_json(url, params)
    if js is None: return None, err
    docs = js.get("documents", [])
    if not docs: return None, "KW_NO_MATCH"
    x, y = docs[0].get("x"), docs[0].get("y")
    if not x or not y: return None, "KW_NO_XY"
    return (float(x), float(y)), None

def kakao_transcoord(x: float, y: float, from_coord="WTM", to_coord="WGS84") -> Tuple[Optional[tuple[float,float]], Optional[str]]:
    """좌표계 변환: WTM->WGS84 등"""
    url = "https://dapi.kakao.com/v2/local/geo/transcoord.json"
    js, err = _call_json(url, {"x": x, "y": y, "input_coord": from_coord, "output_coord": to_coord})
    if js is None: return None, err
    docs = js.get("documents", [])
    if not docs: return None, "TRANSCORD_EMPTY"
    dx, dy = docs[0].get("x"), docs[0].get("y")
    if dx is None or dy is None: return None, "TRANSCORD_NO_XY"
    return (float(dx), float(dy)), None

def detect_and_fix_coord(lon, lat) -> Tuple[Optional[tuple[float,float]], Optional[str]]:
    """좌표계 감지(WGS84/WTM 추정) 후 WGS84 좌표 반환"""
    try:
        lon = float(lon); lat = float(lat)
    except Exception:
        return None, "BAD_NUMBER"
    if _is_wgs84(lon, lat):
        return (lon, lat), None
    if lon > 1000 and lat > 1000:
        conv, err = kakao_transcoord(lon, lat, from_coord="WTM", to_coord="WGS84")
        if conv: return conv, None
        return None, f"WTM_FAIL:{err}"
    return None, "OUT_OF_RANGE"

# =========================
# CSV 로드
# =========================
if not os.path.exists(TARGET_CSV_PATH):
    raise SystemExit(f"'{TARGET_CSV_PATH}' 파일이 없습니다.")

df = pd.read_csv(TARGET_CSV_PATH, encoding="utf-8-sig")

# '법정동' 준비 + 공백→NaN
if '법정동' not in df.columns:
    df['법정동'] = pd.NA
df['법정동'] = df['법정동'].replace(r'^\s*$', pd.NA, regex=True)

# 보조 컬럼도 공백→NaN
for col in ['addr1', 'mapx', 'mapy', 'title']:
    if col in df.columns:
        df[col] = df[col].replace(r'^\s*$', pd.NA, regex=True)

# =========================
# 진행/실패 카운터
# =========================
fail_reason_counter = defaultdict(int)
progress_lock = threading.Lock()
progress_count = 0

def _periodic_save(force: bool=False):
    """N건마다 저장(멀티스레드 안전)"""
    global progress_count
    with progress_lock:
        if force:
            df.to_csv(TARGET_CSV_PATH, index=False, encoding="utf-8-sig");  return
        progress_count += 1
        if progress_count % SAVE_INTERVAL == 0:
            df.to_csv(TARGET_CSV_PATH, index=False, encoding="utf-8-sig")

# =========================
# 0) 좌표 → 역지오코딩 (법정동 빈칸만)
# =========================
if 'mapx' in df.columns and 'mapy' in df.columns:
    mask0 = df['법정동'].isna() & df['mapx'].notna() & df['mapy'].notna()
    if mask0.any():
        idxs = df[mask0].index.tolist()

        def _task0(i):
            lon, lat = df.at[i, 'mapx'], df.at[i, 'mapy']
            fixed, err = detect_and_fix_coord(lon, lat)
            if fixed is None:
                fail_reason_counter[f"COORD_BAD:{err}"] += 1
                return i, None
            addr, err2 = kakao_coord2region(fixed[0], fixed[1])
            if addr is None:
                fail_reason_counter[f"COORD2REGION:{err2}"] += 1
            return i, addr

        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            futures = [ex.submit(_task0, i) for i in idxs]
            for fut in tqdm(as_completed(futures), total=len(futures), desc="[0] 좌표→법정동"):
                i, addr = fut.result()
                if addr and pd.isna(df.at[i, '법정동']):  # 빈칸만 채움
                    df.at[i, '법정동'] = addr
                _periodic_save()
        _periodic_save(force=True)

# =========================
# 1) 주소 → 좌표 → 역지오코딩 (법정동 빈칸 + addr1 있음)
# =========================
if 'addr1' in df.columns:
    mask1 = df['법정동'].isna() & df['addr1'].notna()
    if mask1.any():
        idxs = df[mask1].index.tolist()

        def _task1(i):
            address = df.at[i, 'addr1']
            coord, err = kakao_address_to_coord(address)
            if coord is None:
                fail_reason_counter[f"ADDR2COORD:{err}"] += 1
                return i, None
            addr, err2 = kakao_coord2region(coord[0], coord[1])
            if addr is None:
                fail_reason_counter[f"COORD2REGION:{err2}"] += 1
            return i, addr

        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            futures = [ex.submit(_task1, i) for i in idxs]
            for fut in tqdm(as_completed(futures), total=len(futures), desc="[1] 주소→좌표→법정동"):
                i, addr = fut.result()
                if addr and pd.isna(df.at[i, '법정동']):
                    df.at[i, '법정동'] = addr
                _periodic_save()
        _periodic_save(force=True)

# =========================
# 2) 키워드(title) → 좌표 → 역지오코딩 (법정동 빈칸만)
# =========================
if 'title' in df.columns:
    mask2 = df['법정동'].isna()
    if mask2.any():
        idxs = df[mask2].index.tolist()

        def _task2(i):
            bias = None
            if 'mapx' in df.columns and 'mapy' in df.columns:
                fixed, _ = detect_and_fix_coord(df.at[i,'mapx'], df.at[i,'mapy'])
                if fixed: bias = fixed
            coord, err = kakao_keyword_to_coord(str(df.at[i,'title']), bias=bias)
            if coord is None:
                fail_reason_counter[f"KW2COORD:{err}"] += 1
                return i, None
            addr, err2 = kakao_coord2region(coord[0], coord[1])
            if addr is None:
                fail_reason_counter[f"COORD2REGION:{err2}"] += 1
            return i, addr

        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            futures = [ex.submit(_task2, i) for i in idxs]
            for fut in tqdm(as_completed(futures), total=len(futures), desc="[2] 키워드→좌표→법정동"):
                i, addr = fut.result()
                if addr and pd.isna(df.at[i, '법정동']):
                    df.at[i, '법정동'] = addr
                _periodic_save()
        _periodic_save(force=True)

# =========================
# 3) 좌표 지터링(거친→정밀) (법정동 빈칸 + 좌표 있음)
# =========================
if 'mapx' in df.columns and 'mapy' in df.columns:
    mask3 = df['법정동'].isna() & df['mapx'].notna() & df['mapy'].notna()
    if mask3.any():
        idxs = df[mask3].index.tolist()

        def _task3(i):
            lon, lat = df.at[i, 'mapx'], df.at[i, 'mapy']
            fixed, err = detect_and_fix_coord(lon, lat)
            if fixed is None:
                fail_reason_counter[f"COORD_BAD:{err}"] += 1
                return i, None
            base_lon, base_lat = fixed

            # 3-1) 거친 탐색(4방향)
            for m in range(COARSE_STEP_M, COARSE_MAX_M + 1, COARSE_STEP_M):
                for sx, sy in OFFSETS_4:
                    dx, dy = sx * m, sy * m
                    lon_j = base_lon + _m_to_deg_lon(dx, base_lat)
                    lat_j = base_lat + _m_to_deg_lat(dy)
                    addr, err2 = kakao_coord2region(lon_j, lat_j)
                    if addr:
                        return i, addr

            # 3-2) 정밀 탐색(8방향)
            for m in range(FINE_STEP_M, FINE_MAX_M + 1, FINE_STEP_M):
                for sx, sy in OFFSETS_8:
                    dx, dy = sx * m, sy * m
                    lon_j = base_lon + _m_to_deg_lon(dx, base_lat)
                    lat_j = base_lat + _m_to_deg_lat(dy)
                    addr, err2 = kakao_coord2region(lon_j, lat_j)
                    if addr:
                        return i, addr

            fail_reason_counter["JITTER_FAIL"] += 1
            return i, None

        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            futures = [ex.submit(_task3, i) for i in idxs]
            for fut in tqdm(as_completed(futures), total=len(futures), desc="[3] 좌표 지터링"):
                i, addr = fut.result()
                if addr and pd.isna(df.at[i, '법정동']):
                    df.at[i, '법정동'] = addr
                _periodic_save()
        _periodic_save(force=True)

# =========================
# 결과 요약
# =========================
remain = int(df['법정동'].isna().sum())
print("\n=== 요약 ===")
print("남은 실패 건:", remain)
print("실패 사유 TOP:", sorted(fail_reason_counter.items(), key=lambda x:-x[1])[:10])

# 최종 저장
df.to_csv(TARGET_CSV_PATH, index=False, encoding="utf-8-sig")
print(f"[DONE] 저장 완료: {TARGET_CSV_PATH}")


[2] 키워드→좌표→법정동: 100%|██████████| 12/12 [00:01<00:00,  6.54it/s]



=== 요약 ===
남은 실패 건: 0
실패 사유 TOP: [('COORD_BAD:OUT_OF_RANGE', 35), ('ADDR2COORD:NO_MATCH', 12)]
[DONE] 저장 완료: 관광지_법정동_매핑결과.csv


In [8]:
import pandas as pd

tour_region = '관광지_법정동_매핑결과.csv'
df = pd.read_csv(tour_region)


In [9]:
import numpy as np
import pandas as pd

def minmax_open01(s: pd.Series, eps: float = 1e-6) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    lo, hi = s.min(), s.max()
    if not np.isfinite(lo) or not np.isfinite(hi):  # 전부 NaN인 경우
        return pd.Series(np.nan, index=s.index)
    if hi == lo:  # 값이 전부 같으면 0.5로
        return pd.Series(0.5, index=s.index)
    z = (s - lo) / (hi - lo)          # [0,1]
    return z * (1 - 2*eps) + eps      # (eps, 1-eps)

df["tour_score"] = minmax_open01(df["tour_score"], eps=1e-6)

df.to_csv(tour_region, index=False, encoding="utf-8-sig")